# 02 — Features and CAR targets

Milestone B. Thin consumer of `src.features` and `src.targets`. Inspects the 33-column feature matrix, the 48-target CAR matrix, and confirms no-look-ahead alignment on known event dates (2020-03 COVID Energy, 2022-02 Russia precautionary).

All logic lives in `src/`; this notebook is exploration only.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.io_utils import load_panel
from src.features import build_feature_matrix, check_no_lookahead
from src.targets import build_car_targets, build_sign_targets, check_target_alignment

panel = load_panel()
shocks = pd.read_parquet(Path.cwd().parent / 'outputs' / 'shocks' / 'shocks_v1.parquet')
X = build_feature_matrix(panel, shocks)
Y = build_car_targets(panel)
Y_sign = build_sign_targets(Y)

X.shape, Y.shape, Y_sign.shape

## No-look-ahead verification

In [ ]:
check_no_lookahead(X, panel, shocks), check_target_alignment(Y, panel)

## Feature snapshot at key event months

In [ ]:
events = ['1990-09-30', '2002-10-31', '2008-09-30', '2014-11-30', '2020-03-31', '2022-02-28']
X.loc[events, ['eps_supply', 'eps_agg_demand', 'eps_precaut', 'dom_is_precaut',
                'contamination_flag', 'vix_level', 'vix_regime', 'Recession']].round(2)

## Target distribution by horizon — FF_Enrgy

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, h in zip(axes, [1, 3, 6, 12]):
    ax.hist(Y[('Enrgy', h)].dropna(), bins=40, color='#1f3b66', alpha=0.8)
    ax.set_title(f'FF_Enrgy CAR, h={h}m')
    ax.axvline(0, color='red', lw=0.7)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Feature correlation structure

Shock features should be mutually near-orthogonal; lags of the same shock will show structural correlation.

In [ ]:
shock_cols = [c for c in X.columns if c.startswith('eps_') or c.startswith('dom_') or c.startswith('shock_') or c == 'contamination_flag']
corr = X[shock_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Shock-feature correlation matrix')
plt.tight_layout()
plt.show()

## Missingness report

In [ ]:
miss = X.isna().sum()
miss[miss > 0].to_frame('n_missing').sort_values('n_missing', ascending=False)